## Traffic Video Processing with PyTorch Models

This notebook provides a structured framework to process traffic videos using pre-trained PyTorch models. You can easily adapt it to iterate through multiple models and video files.

In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.8 MB/s eta 0:00:00


In [11]:
import torch
import cv2
import os
from pathlib import Path
import numpy as np
from ultralytics import YOLO
import pandas as pd
from collections import deque

print("Libraries imported successfully.")

Libraries imported successfully.


### 1. Model Loading Function

This function will load a PyTorch model from a given `.pt` file. It's designed to be flexible, allowing you to load different models by simply passing their file paths.

In [3]:
def load_model(model_path: str):
    """Loads an Ultralytics YOLO model and moves it to the appropriate device."""
    if not Path(model_path).is_file():
        raise FileNotFoundError(f"Model file not found at: {model_path}")

    try:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Using device: {device}")
        # Correct way to load Ultralytics YOLO models
        model = YOLO(model_path)
        model.to(device)
        print(f"Model loaded successfully from: {model_path}")
        return model
    except Exception as e:
        print(f"Error loading model {model_path}: {e}")
        return None

### 2. Video Processing Function (Single Video)

This function takes a video file and a loaded model, processes each frame, and can save or display the results. You'll need to customize the `process_frame` part based on what your model does (e.g., object detection, segmentation).

In [19]:
def process_single_video(video_path: str, model, output_video_path: str, config: dict):
    """Processes video with configurable trackers, smoothed trajectories and ID persistence logic."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    colors = {}
    for i, name in model.names.items():
        np.random.seed(i)
        colors[i] = tuple(np.random.randint(0, 255, 3).tolist())

    track_history = {}
    last_seen = {}
    all_data = []
    frame_idx = 0

    # Get tracker config from dict, default to bytetrack
    tracker_type = config.get('tracker', 'bytetrack.yaml')

    while True:
        ret, frame = cap.read()
        if not ret or frame_idx > config.get('end_frame', 99999):
            break

        if frame_idx < config.get('start_frame', 0):
            frame_idx += 1
            continue

        # Apply selected tracker (bytetrack.yaml or botsort.yaml)
        results = model.track(frame, persist=True, tracker=tracker_type, conf=config.get('conf', 0.25), iou=config.get('iou', 0.45), verbose=False)

        if results and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            ids = results[0].boxes.id.int().cpu().tolist()
            clss = results[0].boxes.cls.int().cpu().tolist()
            confs = results[0].boxes.conf.cpu().tolist()

            for box, track_id, cls, conf in zip(boxes, ids, clss, confs):
                last_seen[track_id] = frame_idx
                x1, y1, x2, y2 = map(int, box)
                raw_cx, raw_cy = (x1 + x2) / 2, (y1 + y2) / 2

                if track_id not in track_history:
                    track_history[track_id] = deque(maxlen=30)
                track_history[track_id].append((raw_cx, raw_cy))

                recent_pts = list(track_history[track_id])[-5:]
                cx = sum(p[0] for p in recent_pts) / len(recent_pts)
                cy = sum(p[1] for p in recent_pts) / len(recent_pts)

                name = model.names[cls]
                color = colors.get(cls, (0, 255, 0))

                if config.get('line_thickness', 2) > 0:
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, config['line_thickness'])
                    cv2.putText(frame, f"{name} {track_id}", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
                    points = np.array(list(track_history[track_id]), dtype=np.int32).reshape((-1, 1, 2))
                    cv2.polylines(frame, [points], isClosed=False, color=color, thickness=2)

                all_data.append({
                    'frame': frame_idx, 'track_id': track_id, 'class': name,
                    'conf': conf, 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                    'cx': cx, 'cy': cy
                })

        expired_ids = [tid for tid, last_f in last_seen.items() if frame_idx - last_f > 10]
        for tid in expired_ids:
            track_history.pop(tid, None)
            last_seen.pop(tid, None)

        out.write(frame)
        frame_idx += 1
        if frame_idx % 50 == 0: print(f"Processed {frame_idx} frames using {tracker_type}...")

    cap.release()
    out.release()
    return all_data

### 3. Multiple Videos and Models Orchestration

This function allows you to iterate through a list of video files and apply one or more models to them. This is useful for testing different models or processing a batch of videos.

In [5]:
def process_multiple_videos_with_models(video_paths: list, model_paths: list, output_base_dir: str = 'traffic_analysis_results', display_frames: bool = False):
    """Processes multiple videos using multiple models."""
    for model_idx, model_path in enumerate(model_paths):
        print(f"\n--- Loading and processing with Model: {Path(model_path).name} ---")
        model = load_model(model_path)
        if model is None:
            print(f"Skipping model {model_path} due to loading error.")
            continue

        current_model_output_dir = os.path.join(output_base_dir, f"model_{model_idx}_{Path(model_path).stem}")
        Path(current_model_output_dir).mkdir(parents=True, exist_ok=True)

        for video_path in video_paths:
            print(f"Processing video: {Path(video_path).name}")
            process_single_video(video_path, model, output_dir=current_model_output_dir, display_frames=display_frames)

### 4. Main Execution Block

This is where you define your model paths and video paths, and then call the main processing function. Replace the placeholder paths with your actual file locations.

In [30]:
if __name__ == '__main__':
    CONFIG = {
        "start_frame": 0,
        "end_frame": 100,
        "conf": 0.15,
        "iou": 0.45,
        "line_thickness": 2,
        "tracker": "bytetrack.yaml"  # Options: "bytetrack.yaml", "botsort.yaml"
    }

    MODEL_PATH = '/content/UVH-26-MV-YOLOv11-S.pt'
    VIDEO_PATH = '/content/pixabay_4k.mp4'
    OUT_VIDEO = '/content/annotated_traffic_small1.mp4'
    OUT_EXCEL = '/content/traffic_data_small1.xlsx'

    model = load_model(MODEL_PATH)
    if model:
        data = process_single_video(VIDEO_PATH, model, OUT_VIDEO, CONFIG)
        if data:
            df = pd.DataFrame(data)
            df.to_excel(OUT_EXCEL, index=False)
            display(df.head())
            print(f"Done! Video: {OUT_VIDEO}, Excel: {OUT_EXCEL}")

Using device: cuda
Model loaded successfully from: /content/UVH-26-MV-YOLOv11-S.pt
Processed 50 frames using bytetrack.yaml...
Processed 100 frames using bytetrack.yaml...


,frame,track_id,class,conf,x1,y1,x2,y2,cx,cy
0,0,1,Two-wheeler,0.845638,1994,667,2051,769,2022.5,718.0
1,0,2,LCV,0.837526,2611,859,2862,1120,2736.5,989.5
2,0,3,Two-wheeler,0.762488,1990,870,2060,1002,2025.0,936.0
3,0,4,Two-wheeler,0.758998,2357,749,2421,864,2389.0,806.5
4,0,5,Two-wheeler,0.747743,1891,669,1946,767,1918.5,718.0


Done! Video: /content/annotated_traffic_small1.mp4, Excel: /content/traffic_data_small1.xlsx


In [31]:
# To download the annotated video, use the following code:
from google.colab import files

# Ensure the path matches the one used in the processing step
# It was defined as output_annotated_video_file in the main execution block (cell 8acbc587)
print(f"Attempting to download: {OUT_VIDEO}")
files.download(OUT_VIDEO)

Attempting to download: /content/annotated_traffic_small1.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Next Steps:

1.  **Replace Placeholders**: Update `my_model_paths` and `my_video_paths` in the `Main Execution Block` with your actual `.pt` model files and `.mp4` video files.
2.  **Customize `process_single_video`**: Modify the "--- Model Inference (Customize this part) ---" section within the `process_single_video` function to integrate your model's specific input and output requirements (e.g., preprocessing frames, running inference, drawing bounding boxes/segmentations). Remember that the `load_model` function already moves the model to the GPU if available, so ensure your input tensors are also moved to the same device (e.g., `input_tensor.to(device)`).
3.  **Error Handling and Logging**: You might want to add more robust error handling and logging to the functions.
4.  **Output Format**: Currently, it saves processed videos as `.mp4`. You can change this or add functionality to save detection logs (e.g., CSV, JSON) if needed.
5.  **GPU Usage**: The code now automatically detects and utilizes the GPU (`cuda`) if available. Ensure your Colab runtime is set to a GPU type (like T4 GPU).